# E09 · Bridge a warehouse and an ontology

**Outcome:** Reconcile SQL and SPARQL at the same grain, with explicit treatment of identifiers and missing values.

**Time:** about 50 minutes. Run cells in order. This is the executed solution edition.

The relational table is useful evidence, not an obstacle to semantic modeling. Start with the table's grain and candidate keys. Here the encounter_id identifies one row. The person and encounter are modeled separately, and counts are cast only after reading the source codes as strings. A type-2 slowly changing dimension row would need its own version identity as well as a link to the enduring entity.

SQL constraints and OWL axioms have different roles. A database unique constraint rejects duplicate keys; OWL functionality can imply that two fillers denote the same individual. SQL NULL and absent RDF values are not interchangeable truth values. The adapter must choose what a missing source token means. Aggregation counts solution rows in a selected graph; logical entailment decides whether a proposition follows.

Use reconciliation to discover hidden modeling changes. Compare the same population, filters, time window, units and code versions. A SQL count on source rows and a SPARQL count after a one-to-many label join can disagree even when neither parser reports an error. R2RML gives a standard mapping vocabulary; the supplied mapping file illustrates the subject template, logical table and predicate-object maps. The Python adapter is the executed implementation, not an R2RML processor.

In [1]:
from pathlib import Path
import sys, json
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ontology_lab").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Open this notebook from the extracted course folder.")
from ontology_lab import *
print("Course:", ROOT.name, "| source rows:", len(rows()))

Course: enterprise_ontology_tutorial | source rows: 60


## Ask equivalent SQL and SPARQL questions

In [2]:
db=sql_connection()
sql=dict(db.execute('SELECT diag_1, COUNT(*) FROM encounters GROUP BY diag_1'))
g=build_asserted()
sparql={str(c).rsplit('/',1)[-1]:int(n) for c,n in query(g,'SELECT ?c (COUNT(?r) AS ?n) WHERE { ?r a ex:EncounterRecord; ex:primaryCode ?c } GROUP BY ?c')}
assert sql==sparql=={'250.02':20,'428':20,'493':20}
display(sql)

{'250.02': 20, '428': 20, '493': 20}

## Inspect the declared R2RML bridge

In [3]:
print((ROOT/'ontology/encounter-mapping.ttl').read_text())
print('Physical key -> record IRI:',rows()[0]['encounter_id'],record_iri(rows()[0]))

@prefix rr: <http://www.w3.org/ns/r2rml#> .
@prefix ex: <https://example.org/health/ontology/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
<#EncounterRecordMap> a rr:TriplesMap ;
 rr:logicalTable [ rr:tableName "encounters" ] ;
 rr:subjectMap [ rr:template "https://example.org/health/resource/record/{encounter_id}" ; rr:class ex:EncounterRecord ] ;
 rr:predicateObjectMap [ rr:predicate ex:encounterId ; rr:objectMap [ rr:column "encounter_id" ; rr:datatype xsd:string ] ] ;
 rr:predicateObjectMap [ rr:predicate ex:primaryCode ; rr:objectMap [ rr:template "https://example.org/health/icd9-source/{diag_1}" ; rr:termType rr:IRI ] ] ;
 rr:predicateObjectMap [ rr:predicate ex:stayDays ; rr:objectMap [ rr:column "time_in_hospital" ; rr:datatype xsd:integer ] ] .
# Specification exercise: run ontology_lab.build_asserted for the executable
# Python adapter. This package does not ship an R2RML execution engine.

Physical key -> record IRI: 2595612 https://example.org/health/resource/recor

## Your turn

Write SQL to count source rows with time_in_hospital at least 7; compare with a SPARQL FILTER on ex:stayDays. Return a pair of equal counts.

Replace `answer = None` with your code. A skipped exercise is reported as incomplete; it is not a pass.

In [4]:
answer = (sql_connection().execute('SELECT COUNT(*) FROM encounters WHERE time_in_hospital >= 7').fetchone()[0], int(list(query(build_asserted(),'SELECT (COUNT(?r) AS ?n) WHERE { ?r ex:stayDays ?d FILTER(?d >= 7) }'))[0][0]))

In [5]:
learner_check(answer, lambda x: len(x)==2 and x[0]==x[1] and x[0]>0, 'Both counts must use encounters, not patients.')

Exercise passed.
Out[0]: True


## Explain your model

How would a join to several alternate labels change a naive COUNT(?r)?

Write a short answer below. Check the relevant chapter in the book before promoting a model change.

**My explanation:** Compare the proof premises, source scope and query contract described above; use your own words in a peer review.